# 基于 LangChain 构建 RAG 系统
使用课程笔记 PDF 构建检索增强生成（RAG）系统，让大模型基于资料回答问题。

In [ ]:
import os

os.environ["DEEPSEEK_API_KEY"] = ""

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("note_data.pdf")
docs = loader.load()

print(f"共加载 {len(docs)} 页")
print(f"第一页内容预览:\n{docs[0].page_content[:500]}")

共加载 267 页
第一页内容预览:
⼩⻩搞AI-LLM ⾯试问题
⽬录
1.Transformer前馈神经⽹络⽤的是什么激活函数?
2.解释-下Lora的原理
3.Lora是怎样进⾏初始化矩阵的
4.解释-下AdaLora和QLora的原理
5.解释-下Adapter
6.介绍⼏种常⻅的Adapter
7.解释-下prefix-tuning
8.解释-下P-tuning
9.解释-下Prompt-tuning
10.解释⼀下混合精度的原理
11.训练的时候⽤float16、bfloat16还是float32，为什么?
12.怎么解决训练使⽤float16导致溢出的问题
13.CLM 和M LM 分别是什么，有什么区别?
14.GLM 的⾃回归空⽩填充⽅法与BERT中的使⽤遮蔽语⾔模型(M LM )有什么不同，为⾃然语⾔理解
(NLU)任务带来了哪些优势
15.了解迁移学习吗?⼤模型中是怎么运⽤迁移学习的?
16.Encoder-only、Decoder-only和Encoder-Decoder的模型分别有什么区别，怎么运⽤?
17.为什么现在的⼤语⾔模型都⽤Decoder-only



### 文本拆分
将长文档切分为小块（chunk），便于后续向量化和精确检索。

In [5]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

splits = text_splitter.split_documents(docs)

print(f"共切分为 {len(splits)} 个文本块")
print(f"\n示例文本块:\n{splits[0].page_content}")

共切分为 265 个文本块

示例文本块:
⼩⻩搞AI-LLM ⾯试问题
⽬录
1.Transformer前馈神经⽹络⽤的是什么激活函数?
2.解释-下Lora的原理
3.Lora是怎样进⾏初始化矩阵的
4.解释-下AdaLora和QLora的原理
5.解释-下Adapter
6.介绍⼏种常⻅的Adapter
7.解释-下prefix-tuning
8.解释-下P-tuning
9.解释-下Prompt-tuning
10.解释⼀下混合精度的原理
11.训练的时候⽤float16、bfloat16还是float32，为什么?
12.怎么解决训练使⽤float16导致溢出的问题
13.CLM 和M LM 分别是什么，有什么区别?
14.GLM 的⾃回归空⽩填充⽅法与BERT中的使⽤遮蔽语⾔模型(M LM )有什么不同，为⾃然语⾔理解
(NLU)任务带来了哪些优势
15.了解迁移学习吗?⼤模型中是怎么运⽤迁移学习的?
16.Encoder-only、Decoder-only和Encoder-Decoder的模型分别有什么区别，怎么运⽤?
17.为什么现在的⼤语⾔模型都⽤Decoder-only
18.简述-下Transformer的基本结构和原理
19.Transformer为什么使⽤多头注意⼒机制?
20.Transformer为何让Q(查询)和K(键)使⽤独⽴的权重矩阵进⾏计算，为什么需要Q、K、V(查询、
键、值)三个矩阵?
21.为什么在attention中要进⾏scaled(为什么除以\sqrt{d_k})?
22.Transformer的位置编码作⽤及局限性?
23.为什么Transformer⽤LayerNorm⽽不⽤BatchNorm?
24.解释-下KVcache的原理


### 创建 Embedding 模型 + 向量数据库
使用 HuggingFace 的免费本地 Embedding 模型将文本块向量化，存入 Chroma 向量数据库。

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f"向量数据库创建完成，共存入 {vectorstore._collection.count()} 个文本块")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

向量数据库创建完成，共存入 265 个文本块


## Step 6: 创建检索器

In [8]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

test_results = retriever.invoke("什么是LoRA?")
print(f"检索到 {len(test_results)} 个相关文本块")
for i, doc in enumerate(test_results):
    print(f"\n--- 文本块 {i+1} (来自第 {doc.metadata.get('page', '?')} 页) ---")
    print(doc.page_content[:200])

检索到 4 个相关文本块

--- 文本块 1 (来自第 12 页) ---
1.HoulsbyAdapter：这是最早的Adapter之⼀，设计在Transformer的每⼀层注意⼒块和前馈⽹络
之间插⼊Adapter模块。
2.ParallelAdapter：不同于将Adapter插⼊到主⽹络中，平⾏结构的Adapter与Transformer层并⾏
⼯作，类似残差结构，可以在主⼲⽹络和Adapter之间进⾏特征融合。
3.Com pacter：这是⼀种在减少参

--- 文本块 2 (来自第 257 页) ---
100.2 如何⽤PPL来评估模型？
🫐在预训练阶段：我们通常会准备⼀些“开发集”（DevSet），这其中可包含百科⽂本、新
闻、代码⽚段等。每天或每隔⼀定步数，就让模型在这些测试集上做⼀次前向推理，计算
PPL，看它是否持续下降。
当PPL下降到某个稳定⽔平后，说明模型对通⽤语料的学习已经到达了瓶颈或接近饱和。
需要注意：PPL只适合同⼀模型或同⼀家tokenizer上做

--- 文本块 3 (来自第 258 页) ---
• 祖先追溯挑战：类似于复杂的家谱/⾎缘/亲属关系推理，需要多层逻辑串联。通过设计
“亲属关系针”，测试LLM 处理真实⻓⽂本中多层逻辑挑战的能⼒。在ATC任务中，通过
⼀系列逻辑推理问题，检验模型对⻓⽂本中每个细节的记忆和分析能⼒，在此任务中，我
们去掉了⽆关⽂本(Haystack)的设定，⽽是将所有⽂本设计为关键信息，LLM 必须综合运
⽤⻓⽂本中的所有内容和推理才能准确回答问题。
通过这种考

--- 文本块 4 (来自第 78 页) ---
要的，哪些信息可以被过滤掉。这样做可以使⽹
络在处理输⼊信息时更有针对性，增强表达能
⼒。
避免信息过载
通过⻔控机制的筛选，可以防⽌不必要的信
息进⼊下⼀层，减轻⽹络的计算负担，同时可能
有助于减缓梯度消失问题。
提⾼模型表现
GLU被认为是⼀种⽐标准的⾮线性激活函数
（如ReLU ）更灵活的机制，因为它结合了线性
变换与⻔控机制。这可以提⾼模型的收敛速度，
增强复杂任务中的表现。
Hada


## Step 7: 构建 RAG Agent
使用 `create_agent` + 检索工具，实现「先检索、再回答」的完整流程。（LangChain v1.x 推荐方式）

In [10]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent

llm = init_chat_model("deepseek:deepseek-chat")

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """根据问题检索相关的笔记内容。"""
    retrieved_docs = vectorstore.similarity_search(query, k=4)
    serialized = "\n\n".join(
        f"来源: 第{doc.metadata.get('page', '?')}页\n内容: {doc.page_content}"
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

system_prompt = (
    "你是一个专业的 AI 学习助手。你可以使用检索工具查找参考资料来回答问题。\n"
    "要求：\n"
    "1. 每次回答前，先使用 retrieve_context 工具检索相关资料\n"
    "2. 回答要准确、详细，尽量结合资料中的原文\n"
    "3. 如果资料中没有相关信息，请如实说明\n"
    "4. 使用中文回答"
)

rag_agent = create_agent(llm, tools=[retrieve_context], system_prompt=system_prompt)

print("RAG Agent 构建完成！")

RAG Agent 构建完成！


### 提问测试

In [ ]:
response = rag_agent.invoke(
    {"messages": [{"role": "user", "content": "解释一下LoRA的原理"}]}
)

from IPython.display import Markdown,display

display(Markdown(response["messages"][-1].content))

基于我检索到的资料，让我为您详细解释LoRA的原理：

## LoRA（Low-Rank Adaptation）原理详解

### 1. 基本概念
LoRA是一种**低秩适应**技术，属于参数高效微调（PEFT）方法的一种。根据资料显示：

> "LoRA (Low-Rank Adaptation)：与Adapter类似的思想，但LoRA通过在注意⼒权重矩阵上进⾏低秩近似来减少训练参数。LoRA也是⼀种插⼊性微调技术，主要在Transformer的⾃注意⼒模块中插⼊低秩矩阵更新。"

### 2. 核心思想
LoRA的核心思想是：**大语言模型的权重矩阵在适应新任务时，其更新具有低秩特性**。这意味着权重矩阵的更新可以表示为两个较小矩阵的乘积：

$$
\Delta W = BA
$$

其中：
- $W \in \mathbb{R}^{d \times k}$ 是原始权重矩阵
- $B \in \mathbb{R}^{d \times r}$ 和 $A \in \mathbb{R}^{r \times k}$ 是低秩矩阵
- $r \ll \min(d, k)$ 是秩（通常很小，如4、8、16等）

### 3. 工作原理

#### 3.1 权重更新公式
在微调过程中，前向传播变为：
$$
h = Wx + \Delta Wx = Wx + BAx
$$

其中：
- $W$ 是冻结的原始预训练权重
- $B$ 和 $A$ 是可训练的低秩矩阵
- $x$ 是输入

#### 3.2 参数效率
假设原始权重矩阵 $W$ 的大小为 $d \times k$，那么：
- 全参数微调需要训练 $d \times k$ 个参数
- LoRA只需要训练 $(d + k) \times r$ 个参数

当 $r$ 很小时（如8），参数量减少非常显著。例如，如果 $d=1024, k=1024$：
- 全参数：1,048,576个参数
- LoRA（r=8）：(1024+1024)×8=16,384个参数
- **参数减少约64倍**

### 4. 实现细节

#### 4.1 应用位置
根据资料，LoRA主要应用于：
> "主要在Transformer的⾃注意⼒模块中插⼊低秩矩阵更新。"

具体来说，通常应用于：
- Query、Key、Value投影矩阵
- 输出投影矩阵
- 前馈网络中的某些权重

#### 4.2 初始化策略
- 矩阵 $A$ 通常用随机高斯分布初始化
- 矩阵 $B$ 通常初始化为零矩阵
- 这样初始时 $\Delta W = 0$，不会改变原始模型的行为

### 5. 优势特点

#### 5.1 计算效率
- **内存占用少**：只需要存储和更新少量参数
- **训练速度快**：梯度计算只涉及小矩阵
- **存储效率高**：可以共享基础模型，只存储LoRA权重

#### 5.2 灵活性
- **多任务适配**：可以为不同任务训练不同的LoRA权重
- **组合使用**：多个LoRA权重可以叠加使用
- **易于部署**：推理时可以将LoRA权重合并到基础权重中

#### 5.3 性能表现
- 通常能达到接近全参数微调的性能
- 在某些任务上甚至可能表现更好（正则化效应）

### 6. 与其他方法的比较

根据资料中提到的其他Adapter方法：
- **Houlsby Adapter**：在注意力块和前馈网络之间插入Adapter模块
- **Parallel Adapter**：与Transformer层并行工作，类似残差结构
- **Compacter**：使用低秩参数分解技术进一步减少参数量

LoRA与这些方法的主要区别在于：
- **直接修改权重**：LoRA直接在权重矩阵上添加低秩更新
- **更少的额外计算**：推理时可以通过权重合并消除额外计算
- **更好的可扩展性**：更容易扩展到不同的模型架构

### 7. 实际应用

#### 7.1 典型配置
- 秩 $r$：通常为4、8、16、32
- $\alpha$（缩放因子）：控制LoRA更新的强度
- 应用层：通常应用于所有Transformer层或特定层

#### 7.2 使用场景
- **领域适应**：将通用模型适配到特定领域
- **任务微调**：针对特定下游任务优化
- **多语言适配**：扩展模型到新语言
- **风格迁移**：调整模型的生成风格

### 8. 总结

LoRA的核心创新在于发现了**大模型权重更新的低秩特性**，并利用这一特性实现了：
1. **极高的参数效率**：只需训练原模型参数的0.1%-1%
2. **优秀的性能保持**：能达到接近全参数微调的效果
3. **灵活的部署方式**：支持权重合并和多任务切换
4. **广泛的应用性**：适用于各种大语言模型的微调场景

这种技术使得在有限的计算资源下微调大模型成为可能，极大地降低了AI应用的门槛和成本。

In [12]:
response = rag_agent.invoke(
    {"messages": [{"role": "user", "content": "Transformer为什么使用多头注意力机制？"}]}
)

print("=" * 60)
print("问题: Transformer为什么使用多头注意力机制？")
print("=" * 60)
print("\n回答:")
print(response["messages"][-1].content)

问题: Transformer为什么使用多头注意力机制？

回答:
基于检索到的资料，我来详细解释Transformer为什么使用多头注意力机制：

## Transformer使用多头注意力机制的主要原因

### 1. **增强模型的表达能力**
根据资料显示，多头注意力机制"将注意⼒机制并⾏拆分为多个独⽴的注意⼒头（head），每个head在不同⼦空间捕捉序列依赖，然后再拼接"。这种设计让模型能够**同时关注不同方面的信息**，从而获得更丰富的特征表示。

### 2. **并行捕捉不同类型的依赖关系**
多头注意力允许模型**并行地学习多种注意力模式**：
- 每个注意力头可以关注序列中不同位置的关系
- 不同头可以捕捉不同类型的语法或语义关系
- 例如，一个头可能关注局部依赖，另一个头可能关注长距离依赖

### 3. **提高模型的灵活性**
从资料中可以看到："在同⼀次注意⼒计算⾥使⽤多个头；每个头有⾃⼰的⼀套投影参数，最后再合并，表达⼒更强"。这意味着：
- 每个头都有自己的参数矩阵（Wᵢᴼ, Wᵢᴷ, Wᵢⱽ）
- 模型可以学习到更复杂的注意力模式
- 不同头可以专门化处理不同类型的特征

### 4. **技术实现细节**
根据资料中的描述：
- **BERT-base示例**："每个⾃注意⼒头的输出维度是64（总维度为768），⽽多头注意⼒机制则是将12个⾃注意⼒头的输出拼接(concatenate)后，再通过⼀个线性层处理"
- **计算过程**：Multi-Head Self-Attention将多个不同单头的Self-Attention输出Concat成⼀条，然后再经过⼀个全连接层降维输出

### 5. **与单头注意力的对比优势**
多头注意力相比单头注意力的优势：
- **更强的表示能力**：多个头可以并行学习不同的特征表示
- **更好的泛化能力**：通过多个头的组合，模型能够更好地处理复杂的序列关系
- **计算效率**：虽然增加了参数，但通过并行计算可以保持较高的计算效率

### 6. **实际应用效果**
资料指出："绝⼤多数Transform er都使⽤多头"，这说明了多头注意力机制在实际应用中的广泛接受度和有效性。它被应用于：
- NLP任务（如BERT、GPT系列）
- 计算机视觉任务
- 推荐系统
- 语音处理等各

### 自由提问
修改下面的 `question` 变量，向你的笔记资料提出任何问题。

In [ ]:
question = "在这里输入你的问题"

response = rag_agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)
print(response["messages"][-1].content)

## 附录：下次使用时直接加载已有向量数据库
首次运行完成后，`chroma_db` 文件夹会保存在本地。之后无需重新导入 PDF，直接加载即可。

In [ ]:
# 下次使用时，跳过 Step 3-5，直接运行以下代码加载已有的向量数据库

# from langchain_huggingface import HuggingFaceEmbeddings
# from langchain_chroma import Chroma
#
# embeddings = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"
# )
#
# vectorstore = Chroma(
#     persist_directory="./chroma_db",
#     embedding_function=embeddings
# )
#
# retriever = vectorstore.as_retriever(search_kwargs={"k": 4})